
# Buy Fee Model V12A.3 — Working Buyer DiD Pipeline

## Purpose

This notebook is designed for the production buyer file:

`biweekly_buyer_churn_flags_detail_reactivated_loop1.csv`

which has more than 6.6 million rows.

It fixes the two major issues found in earlier V12A versions:

1. **Memory safety** — the raw buyer CSV is never loaded fully into memory. It is read in chunks and immediately aggregated to `Location × Biweekly Period`.
2. **Correct event dates** — buyer windows use the real `BUYFEE_CHANGE_DATE` from the V6.1 strict event file, not `Corporate_Change_ID` interpreted as a date.

## Buyer event windows

For each fee event:

- PRE = 56 calendar days before the true Buy Fee change date
- POST = 56 calendar days after the true Buy Fee change date
- A biweekly buyer record is included when its period overlaps the PRE or POST window

The notebook automatically tests 3/3, 2/2, and 1/1 PRE/POST buyer-period coverage and uses the strongest available rule while clearly labeling the result.

## Matched-control Buyer DiD

For each treated fee-change event and matched control:

`Buyer DiD = Treated PRE→POST change - Control PRE→POST change`

Metrics:

- Unique Buyers
- Active Buyers
- Fallout Buyers
- Dark Buyers
- Reactivated Buyers
- Net New Buyers
- Days Since Last Purchase
- Days Since Previous Location Purchase

The final outputs are used to decide whether PRE-event buyer features should enter the next predictive model.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import gc
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================

DATA_FOLDER = Path(
    r"C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion"
)

V61_DIR = DATA_FOLDER / 'buy_fee_model_dataset_v6_1'
V82_DIR = DATA_FOLDER / 'buy_fee_model_v8_2_outputs'

BUYER_FILE = DATA_FOLDER / 'biweekly_buyer_churn_flags_detail_reactivated_loop1.csv'
STRICT_EVENT_FILE = V61_DIR / '09_MODEL_READY_STRICT_V6_1.csv'
MATCHED_PAIRS_FILE = V82_DIR / '03_MATCHED_CONTROL_PAIRS.csv'
DID_FILE = V82_DIR / '04_DID_EVENT_LEVEL_RESULTS.csv'

OUTPUT_DIR = DATA_FOLDER / 'buy_fee_model_v12a_3_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BUYER_AGG_FILE = OUTPUT_DIR / '01_BUYER_BIWEEKLY_AGGREGATE.csv'

CHUNK_SIZE = 75_000
SCENARIOS = [10, 15, 20]
SCENARIO_TOLERANCE = 2.5
PRE_DAYS = 56
POST_DAYS = 56

RPU_SUCCESS_MIN = 0.0
CONVERSION_SAFE_MIN_PP = -2.0
VOLUME_SAFE_MIN_PCT = -0.10

print('=' * 80)
print('V12A.3 INPUT CHECK')
print('=' * 80)

for p in [BUYER_FILE, STRICT_EVENT_FILE, MATCHED_PAIRS_FILE, DID_FILE]:
    print(f'{p.name:<55} {p.exists()}')

missing = [p for p in [BUYER_FILE, STRICT_EVENT_FILE, MATCHED_PAIRS_FILE, DID_FILE] if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required file(s):\n' + '\n'.join(str(p) for p in missing))

print('\nOutput folder:')
print(OUTPUT_DIR)


V12A.3 INPUT CHECK
biweekly_buyer_churn_flags_detail_reactivated_loop1.csv True
09_MODEL_READY_STRICT_V6_1.csv                          True
03_MATCHED_CONTROL_PAIRS.csv                            True
04_DID_EVENT_LEVEL_RESULTS.csv                          True

Output folder:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v12a_3_outputs


## 1. Inspect buyer columns and resolve field names

In [2]:

header = pd.read_csv(
    BUYER_FILE,
    nrows=5,
    low_memory=True,
    encoding='utf-8',
    encoding_errors='replace'
)

print('Buyer columns:')
for c in header.columns:
    print(' -', c)

def resolve_col(columns, aliases):
    lookup = {str(c).strip().upper(): c for c in columns}
    for alias in aliases:
        key = alias.strip().upper()
        if key in lookup:
            return lookup[key]
    return None

COL = {
    'location': resolve_col(header.columns, ['VRLOCATION', 'LOCATION']),
    'buyer': resolve_col(header.columns, ['VRBUYER', 'BUYER']),
    'start_date': resolve_col(header.columns, ['BiWeekly_Start_Date', 'BIWEEKLY_START_DATE', 'BIWEEKLY_START']),
    'end_date': resolve_col(header.columns, ['BiWeekly_End_Date', 'BIWEEKLY_END_DATE', 'BIWEEKLY_END']),
    'active': resolve_col(header.columns, ['Active']),
    'fallout': resolve_col(header.columns, ['Fallout']),
    'dark': resolve_col(header.columns, ['Dark']),
    'reactivated': resolve_col(header.columns, ['Reactivated']),
    'net_new': resolve_col(header.columns, ['Net_New', 'Net New']),
    'days_since_last': resolve_col(header.columns, ['Days_Since_Last_Purchase']),
    'days_since_prev_loc': resolve_col(header.columns, ['Days_Since_Previous_Location_Purchase']),
}

mapping = pd.DataFrame([{'Logical_Field': k, 'Resolved_Column': v} for k, v in COL.items()])
display(mapping)

required = ['location', 'buyer', 'start_date', 'end_date']
missing = [k for k in required if COL[k] is None]
if missing:
    raise ValueError('Missing required buyer fields: ' + ', '.join(missing))


Buyer columns:
 - VRLOCATION
 - VRBUYER
 - BiWeekly_ID
 - BiWeekly_Start_Date
 - BiWeekly_End_Date
 - first_location_sale_date
 - Last_Purchase_Date
 - Previous_Location_Purchase_Date
 - Days_Since_Previous_Location_Purchase
 - Last_Purchase_Was_Reactivation
 - Days_Since_Last_Purchase
 - Lifecycle_Status
 - Active
 - Fallout
 - Dark
 - Reactivated
 - Net_New


,Logical_Field,Resolved_Column
0,location,VRLOCATION
1,buyer,VRBUYER
2,start_date,BiWeekly_Start_Date
3,end_date,BiWeekly_End_Date
4,active,Active
5,fallout,Fallout
6,dark,Dark
7,reactivated,Reactivated
8,net_new,Net_New
9,days_since_last,Days_Since_Last_Purchase


## 2. Chunk the 6.6M+ buyer file and aggregate to Location × Biweekly Period

In [3]:

USECOLS = [c for c in [
    COL['location'], COL['buyer'], COL['start_date'], COL['end_date'],
    COL['active'], COL['fallout'], COL['dark'], COL['reactivated'], COL['net_new'],
    COL['days_since_last'], COL['days_since_prev_loc']
] if c is not None]


def clean_text(s):
    return s.astype('string').fillna('').str.strip()


def to_flag(s):
    num = pd.to_numeric(s, errors='coerce')
    out = pd.Series(np.zeros(len(s), dtype=np.int8), index=s.index)
    numeric_mask = num.notna()
    if numeric_mask.any():
        out.loc[numeric_mask] = num.loc[numeric_mask].gt(0).astype('int8')
    text_mask = ~numeric_mask
    if text_mask.any():
        txt = s.loc[text_mask].astype('string').fillna('').str.strip().str.upper()
        true_values = {'1','Y','YES','TRUE','ACTIVE','FALLOUT','DARK','REACTIVATED','NET NEW','NET_NEW'}
        out.loc[text_mask] = txt.isin(true_values).astype('int8')
    return out

# Chunk-level aggregates are very small compared with raw data.
parts = []
rows_read = 0

reader = pd.read_csv(
    BUYER_FILE,
    usecols=USECOLS,
    chunksize=CHUNK_SIZE,
    low_memory=True,
    encoding='utf-8',
    encoding_errors='replace'
)

for chunk_num, chunk in enumerate(reader, start=1):
    rows_read += len(chunk)

    loc = clean_text(chunk[COL['location']]).str.upper()
    buyer_id = clean_text(chunk[COL['buyer']])
    start = pd.to_datetime(chunk[COL['start_date']], errors='coerce')
    end = pd.to_datetime(chunk[COL['end_date']], errors='coerce')

    valid = loc.ne('') & buyer_id.ne('') & start.notna() & end.notna()
    if not valid.any():
        del chunk, loc, buyer_id, start, end, valid
        gc.collect()
        continue

    temp = pd.DataFrame({
        'VRLOCATION': loc.loc[valid].to_numpy(),
        'VRBUYER': buyer_id.loc[valid].to_numpy(),
        'BiWeekly_Start': start.loc[valid].to_numpy(),
        'BiWeekly_End': end.loc[valid].to_numpy(),
    })

    for logical, outcol in [
        ('active','Active_Flag'),
        ('fallout','Fallout_Flag'),
        ('dark','Dark_Flag'),
        ('reactivated','Reactivated_Flag'),
        ('net_new','Net_New_Flag'),
    ]:
        source = COL.get(logical)
        if source is not None:
            temp[outcol] = to_flag(chunk.loc[valid, source]).to_numpy()
        else:
            temp[outcol] = np.int8(0)

    if COL['days_since_last'] is not None:
        temp['Days_Since_Last'] = pd.to_numeric(
            chunk.loc[valid, COL['days_since_last']], errors='coerce'
        ).astype('float32').to_numpy()
    else:
        temp['Days_Since_Last'] = np.nan

    if COL['days_since_prev_loc'] is not None:
        temp['Days_Since_Prev_Loc'] = pd.to_numeric(
            chunk.loc[valid, COL['days_since_prev_loc']], errors='coerce'
        ).astype('float32').to_numpy()
    else:
        temp['Days_Since_Prev_Loc'] = np.nan

    a = (
        temp.groupby(['VRLOCATION','BiWeekly_Start','BiWeekly_End'], dropna=False, sort=False)
        .agg(
            Unique_Buyers=('VRBUYER','nunique'),
            Active_Buyers=('Active_Flag','sum'),
            Fallout_Buyers=('Fallout_Flag','sum'),
            Dark_Buyers=('Dark_Flag','sum'),
            Reactivated_Buyers=('Reactivated_Flag','sum'),
            Net_New_Buyers=('Net_New_Flag','sum'),
            Days_Since_Last_Sum=('Days_Since_Last','sum'),
            Days_Since_Last_Count=('Days_Since_Last','count'),
            Days_Since_Prev_Loc_Sum=('Days_Since_Prev_Loc','sum'),
            Days_Since_Prev_Loc_Count=('Days_Since_Prev_Loc','count'),
        )
        .reset_index()
    )

    parts.append(a)

    if chunk_num % 10 == 0:
        print(f'Chunks={chunk_num:,} | rows read={rows_read:,}')

    del chunk, loc, buyer_id, start, end, valid, temp, a
    gc.collect()

print('\nRaw buyer rows read:', f'{rows_read:,}')
print('Partial aggregate chunks:', len(parts))


Chunks=10 | rows read=750,000
Chunks=20 | rows read=1,500,000
Chunks=30 | rows read=2,250,000
Chunks=40 | rows read=3,000,000
Chunks=50 | rows read=3,750,000
Chunks=60 | rows read=4,500,000
Chunks=70 | rows read=5,250,000
Chunks=80 | rows read=6,000,000
Chunks=90 | rows read=6,679,333

Raw buyer rows read: 6,679,333
Partial aggregate chunks: 90


## 3. Combine chunk aggregates

In [4]:

raw_agg = pd.concat(parts, ignore_index=True)
del parts
gc.collect()

buyer = (
    raw_agg.groupby(['VRLOCATION','BiWeekly_Start','BiWeekly_End'], dropna=False, sort=False)
    .agg(
        Unique_Buyers=('Unique_Buyers','sum'),
        Active_Buyers=('Active_Buyers','sum'),
        Fallout_Buyers=('Fallout_Buyers','sum'),
        Dark_Buyers=('Dark_Buyers','sum'),
        Reactivated_Buyers=('Reactivated_Buyers','sum'),
        Net_New_Buyers=('Net_New_Buyers','sum'),
        Days_Since_Last_Sum=('Days_Since_Last_Sum','sum'),
        Days_Since_Last_Count=('Days_Since_Last_Count','sum'),
        Days_Since_Prev_Loc_Sum=('Days_Since_Prev_Loc_Sum','sum'),
        Days_Since_Prev_Loc_Count=('Days_Since_Prev_Loc_Count','sum'),
    )
    .reset_index()
)

del raw_agg
gc.collect()

buyer['Avg_Days_Since_Last_Purchase'] = np.where(
    buyer['Days_Since_Last_Count'] > 0,
    buyer['Days_Since_Last_Sum'] / buyer['Days_Since_Last_Count'],
    np.nan
)

buyer['Avg_Days_Since_Previous_Location_Purchase'] = np.where(
    buyer['Days_Since_Prev_Loc_Count'] > 0,
    buyer['Days_Since_Prev_Loc_Sum'] / buyer['Days_Since_Prev_Loc_Count'],
    np.nan
)

buyer.to_csv(BUYER_AGG_FILE, index=False)

print('Buyer biweekly aggregate rows:', f'{len(buyer):,}')
print('Buyer locations:', buyer['VRLOCATION'].nunique())
print('Buyer date range:', buyer['BiWeekly_Start'].min(), 'to', buyer['BiWeekly_End'].max())
print('Saved:', BUYER_AGG_FILE)


Buyer biweekly aggregate rows: 3,498
Buyer locations: 49
Buyer date range: 2019-12-30 00:00:00 to 2022-09-24 00:00:00
Saved: C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v12a_3_outputs\01_BUYER_BIWEEKLY_AGGREGATE.csv


## 4. Load the TRUE Buy Fee change date from V6.1 and attach it to V8.2 matched pairs

In [5]:

strict_header = pd.read_csv(STRICT_EVENT_FILE, nrows=0)
required_strict = ['Segment_Change_ID','BUYFEE_CHANGE_DATE']
missing = [c for c in required_strict if c not in strict_header.columns]
if missing:
    raise ValueError('Strict event file missing: ' + ', '.join(missing))

strict_dates = pd.read_csv(
    STRICT_EVENT_FILE,
    usecols=required_strict,
    low_memory=True
)

strict_dates['BUYFEE_CHANGE_DATE'] = pd.to_datetime(
    strict_dates['BUYFEE_CHANGE_DATE'], errors='coerce'
)

strict_dates = (
    strict_dates.dropna(subset=['Segment_Change_ID','BUYFEE_CHANGE_DATE'])
    .drop_duplicates(subset=['Segment_Change_ID'])
)

pairs = pd.read_csv(MATCHED_PAIRS_FILE, low_memory=True)

for c in ['Change_Date','BUYFEE_CHANGE_DATE']:
    if c in pairs.columns:
        pairs.drop(columns=[c], inplace=True)

pairs = pairs.merge(
    strict_dates,
    on='Segment_Change_ID',
    how='left',
    validate='many_to_one'
)

pairs.rename(columns={'BUYFEE_CHANGE_DATE':'Change_Date'}, inplace=True)

for c in ['Treated_Location','Control_Location']:
    pairs[c] = pairs[c].astype('string').fillna('').str.strip().str.upper()

pairs['BUYFEE_CHANGE_AMOUNT'] = pd.to_numeric(pairs['BUYFEE_CHANGE_AMOUNT'], errors='coerce')

scenario_mask = pd.Series(False, index=pairs.index)
for fee in SCENARIOS:
    scenario_mask |= pairs['BUYFEE_CHANGE_AMOUNT'].between(
        fee - SCENARIO_TOLERANCE,
        fee + SCENARIO_TOLERANCE
    )

pairs = pairs[
    scenario_mask
    & pairs['Change_Date'].notna()
    & pairs['Treated_Location'].ne('')
    & pairs['Control_Location'].ne('')
].copy()

print('Matched pair rows:', f'{len(pairs):,}')
print('Treated events:', pairs['Segment_Change_ID'].nunique())
print('True fee date range:', pairs['Change_Date'].min(), 'to', pairs['Change_Date'].max())


Matched pair rows: 2,425
Treated events: 810
True fee date range: 2023-02-12 00:00:00 to 2026-02-08 00:00:00


## 5. Date-overlap and location-overlap diagnostic

In [7]:
# ============================================================
# V12A.3 - SAFE DATE / LOCATION OVERLAP DIAGNOSTIC
# ============================================================
#
# IMPORTANT:
# This block does NOT fabricate buyer PRE/POST data when
# buyer history does not overlap the fee-event dates.
#
# It determines whether Buyer DiD is possible and sets:
#
#     BUYER_DID_AVAILABLE = True / False
#
# ============================================================

import pandas as pd
import numpy as np


print("=" * 80)
print("BUYER / BUY FEE DATE & LOCATION OVERLAP")
print("=" * 80)


# ============================================================
# 1. GET AVAILABLE DATE RANGES
# ============================================================

buyer_min = pd.to_datetime(
    buyer["BiWeekly_Start"],
    errors="coerce"
).min()

buyer_max = pd.to_datetime(
    buyer["BiWeekly_End"],
    errors="coerce"
).max()


fee_min = pd.to_datetime(
    pairs["Change_Date"],
    errors="coerce"
).min()

fee_max = pd.to_datetime(
    pairs["Change_Date"],
    errors="coerce"
).max()


# ============================================================
# 2. DATE WINDOW REQUIRED FOR +/- 56 DAY EVENT STUDY
# ============================================================

required_buyer_start = (
    fee_min
    - pd.Timedelta(days=PRE_DAYS)
)

required_buyer_end = (
    fee_max
    + pd.Timedelta(days=POST_DAYS)
)


# ============================================================
# 3. LOCATION SETS
# ============================================================

buyer_locations = set(
    buyer[
        "VRLOCATION"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .unique()
)


treated_locations = set(
    pairs[
        "Treated_Location"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .unique()
)


control_locations = set(
    pairs[
        "Control_Location"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .unique()
)


matched_treated_locations = (
    treated_locations
    & buyer_locations
)

matched_control_locations = (
    control_locations
    & buyer_locations
)


# ============================================================
# 4. CHECK EACH FEE DATE AGAINST BUYER HISTORY
# ============================================================

pairs[
    "Fee_Date_Inside_Buyer_History"
] = (
    pairs[
        "Change_Date"
    ]
    .between(
        buyer_min,
        buyer_max
    )
)


pairs_inside_history = int(
    pairs[
        "Fee_Date_Inside_Buyer_History"
    ].sum()
)


pair_count = len(
    pairs
)


inside_pct = (
    pairs_inside_history
    / pair_count
    if pair_count > 0
    else np.nan
)


# ============================================================
# 5. BROAD DATE-RANGE OVERLAP
# ============================================================

date_ranges_overlap = bool(
    buyer_min <= fee_max
    and buyer_max >= fee_min
)


# ============================================================
# 6. GAP BETWEEN BUYER HISTORY AND FIRST FEE EVENT
# ============================================================

if (
    pd.notna(buyer_max)
    and pd.notna(fee_min)
):

    gap_days = (
        fee_min
        - buyer_max
    ).days

else:

    gap_days = np.nan


# ============================================================
# 7. DETERMINE WHETHER BUYER DID IS POSSIBLE
# ============================================================

BUYER_DID_AVAILABLE = bool(
    pairs_inside_history > 0
    and date_ranges_overlap
)


# ============================================================
# 8. PRINT DIAGNOSTIC
# ============================================================

print(
    "\nAvailable buyer history:"
)

print(
    buyer_min,
    "to",
    buyer_max
)


print(
    "\nSelected Buy Fee events:"
)

print(
    fee_min,
    "to",
    fee_max
)


print(
    "\nBuyer history REQUIRED for full +/-56-day analysis:"
)

print(
    required_buyer_start,
    "to",
    required_buyer_end
)


print(
    "\nBroad date ranges overlap:",
    date_ranges_overlap
)


print(
    "Matched-pair rows with fee date inside buyer history:",
    f"{pairs_inside_history:,}",
    "/",
    f"{pair_count:,}"
)


print(
    "Percent of matched pairs covered:",
    (
        f"{inside_pct:.2%}"
        if pd.notna(inside_pct)
        else "N/A"
    )
)


print(
    "\nGap from end of buyer data to first fee event:",
    gap_days,
    "days"
)


print(
    "\nTreated locations found in buyer data:",
    len(matched_treated_locations),
    "/",
    len(treated_locations)
)


print(
    "Control locations found in buyer data:",
    len(matched_control_locations),
    "/",
    len(control_locations)
)


# ============================================================
# 9. MISSING LOCATION DETAILS
# ============================================================

missing_treated_locations = sorted(
    treated_locations
    - buyer_locations
)


missing_control_locations = sorted(
    control_locations
    - buyer_locations
)


if missing_treated_locations:

    print(
        "\nTreated locations missing from buyer data:"
    )

    print(
        missing_treated_locations
    )


if missing_control_locations:

    print(
        "\nControl locations missing from buyer data:"
    )

    print(
        missing_control_locations
    )


# ============================================================
# 10. CREATE DIAGNOSTIC OUTPUT
# ============================================================

diagnostic = pd.DataFrame([
    {
        "Metric":
            "Buyer_History_Start",

        "Value":
            buyer_min
    },

    {
        "Metric":
            "Buyer_History_End",

        "Value":
            buyer_max
    },

    {
        "Metric":
            "Fee_Event_Start",

        "Value":
            fee_min
    },

    {
        "Metric":
            "Fee_Event_End",

        "Value":
            fee_max
    },

    {
        "Metric":
            "Required_Buyer_History_Start_For_56_Day_Window",

        "Value":
            required_buyer_start
    },

    {
        "Metric":
            "Required_Buyer_History_End_For_56_Day_Window",

        "Value":
            required_buyer_end
    },

    {
        "Metric":
            "Date_Ranges_Overlap",

        "Value":
            date_ranges_overlap
    },

    {
        "Metric":
            "Matched_Pairs_Inside_Buyer_History",

        "Value":
            pairs_inside_history
    },

    {
        "Metric":
            "Total_Matched_Pairs",

        "Value":
            pair_count
    },

    {
        "Metric":
            "Pct_Matched_Pairs_Inside_Buyer_History",

        "Value":
            inside_pct
    },

    {
        "Metric":
            "Gap_Days_Buyer_End_To_First_Fee_Event",

        "Value":
            gap_days
    },

    {
        "Metric":
            "Treated_Locations_Matched",

        "Value":
            len(
                matched_treated_locations
            )
    },

    {
        "Metric":
            "Control_Locations_Matched",

        "Value":
            len(
                matched_control_locations
            )
    },

    {
        "Metric":
            "Buyer_DiD_Available",

        "Value":
            BUYER_DID_AVAILABLE
    },
])


diagnostic.to_csv(
    OUTPUT_DIR
    / "BUYER_DATE_LOCATION_DIAGNOSTIC.csv",
    index=False
)


# ============================================================
# 11. CREATE SAFE SUBSET IF ANY EVENTS OVERLAP
# ============================================================

if BUYER_DID_AVAILABLE:

    pairs_buyer_eligible = (
        pairs[
            pairs[
                "Fee_Date_Inside_Buyer_History"
            ]
        ]
        .copy()
    )


    print(
        "\n"
        + "=" * 80
    )

    print(
        "BUYER DID CAN PROCEED"
    )

    print(
        "=" * 80
    )


    print(
        "Eligible matched pair rows:",
        f"{len(pairs_buyer_eligible):,}"
    )


    print(
        "Continue to PRE/POST buyer-window creation."
    )


else:

    # Create an empty object so later inspection does not
    # fail simply because the variable is undefined.

    pairs_buyer_eligible = (
        pairs.iloc[
            0:0
        ]
        .copy()
    )


    print(
        "\n"
        + "=" * 80
    )

    print(
        "BUYER DID CANNOT BE CALCULATED"
    )

    print(
        "=" * 80
    )


    print(
        "\nReason:"
    )

    print(
        "The buyer-history period does not overlap "
        "the selected Buy Fee events."
    )


    print(
        "\nThis is a DATA COVERAGE issue, not a coding issue."
    )


    print(
        "\nCurrent buyer data:"
    )

    print(
        f"{buyer_min.date()} through "
        f"{buyer_max.date()}"
    )


    print(
        "\nCurrent fee-event history:"
    )

    print(
        f"{fee_min.date()} through "
        f"{fee_max.date()}"
    )


    print(
        "\nFor the full +/-56 day Buyer DiD study, "
        "buyer data should ideally cover:"
    )

    print(
        f"{required_buyer_start.date()} through "
        f"{required_buyer_end.date()}"
    )


    print(
        "\nDO NOT use the 2019-2022 buyer behavior "
        "as PRE/POST evidence for 2023-2026 fee changes."
    )


print(
    "\nDiagnostic saved to:"
)

print(
    OUTPUT_DIR
    / "BUYER_DATE_LOCATION_DIAGNOSTIC.csv"
)

BUYER / BUY FEE DATE & LOCATION OVERLAP

Available buyer history:
2019-12-30 00:00:00 to 2022-09-24 00:00:00

Selected Buy Fee events:
2023-02-12 00:00:00 to 2026-02-08 00:00:00

Buyer history REQUIRED for full +/-56-day analysis:
2022-12-18 00:00:00 to 2026-04-05 00:00:00

Broad date ranges overlap: False
Matched-pair rows with fee date inside buyer history: 0 / 2,425
Percent of matched pairs covered: 0.00%

Gap from end of buyer data to first fee event: 141 days

Treated locations found in buyer data: 23 / 24
Control locations found in buyer data: 46 / 50

Treated locations missing from buyer data:
['TAMPA']

Control locations missing from buyer data:
['AUGUSTA', 'ELPASO', 'NEWJERSEY', 'TAMPA']

BUYER DID CANNOT BE CALCULATED

Reason:
The buyer-history period does not overlap the selected Buy Fee events.

This is a DATA COVERAGE issue, not a coding issue.

Current buyer data:
2019-12-30 through 2022-09-24

Current fee-event history:
2023-02-12 through 2026-02-08

For the full +/-56 d

## 6. Attach event outcomes

In [8]:

did = pd.read_csv(DID_FILE, low_memory=True)
keep = [c for c in [
    'Segment_Change_ID',
    'DiD_RPU_Change_Dollar',
    'DiD_Conversion_Change_pp',
    'DiD_Volume_Change_Pct',
    'Treated_Lane',
    'Treated_Price_Tier'
] if c in did.columns]

did_small = did[keep].drop_duplicates(subset=['Segment_Change_ID'])
pairs = pairs.merge(did_small, on='Segment_Change_ID', how='left')
del did, did_small
gc.collect()


0

## 7. Build Location × Change Date buyer PRE/POST summaries

In [9]:

METRICS = [
    'Unique_Buyers',
    'Active_Buyers',
    'Fallout_Buyers',
    'Dark_Buyers',
    'Reactivated_Buyers',
    'Net_New_Buyers',
    'Avg_Days_Since_Last_Purchase',
    'Avg_Days_Since_Previous_Location_Purchase',
]

COUNT_METRICS = [
    'Unique_Buyers','Active_Buyers','Fallout_Buyers','Dark_Buyers','Reactivated_Buyers','Net_New_Buyers'
]

needed = pd.concat([
    pairs[['Treated_Location','Change_Date']].rename(columns={'Treated_Location':'Location'}),
    pairs[['Control_Location','Change_Date']].rename(columns={'Control_Location':'Location'})
], ignore_index=True).drop_duplicates()

summary_rows=[]

for idx, r in enumerate(needed.itertuples(index=False), start=1):
    loc = r.Location
    change_date = pd.Timestamp(r.Change_Date)
    d = buyer[buyer['VRLOCATION'].eq(loc)]
    if d.empty:
        continue

    pre_start = change_date - pd.Timedelta(days=PRE_DAYS)
    pre_end = change_date - pd.Timedelta(days=1)
    post_start = change_date + pd.Timedelta(days=1)
    post_end = change_date + pd.Timedelta(days=POST_DAYS)

    pre = d[(d['BiWeekly_Start'] <= pre_end) & (d['BiWeekly_End'] >= pre_start)]
    post = d[(d['BiWeekly_Start'] <= post_end) & (d['BiWeekly_End'] >= post_start)]

    row = {
        'Location': loc,
        'Change_Date': change_date,
        'Pre_Periods': pre['BiWeekly_Start'].nunique(),
        'Post_Periods': post['BiWeekly_Start'].nunique(),
    }

    for metric in METRICS:
        pre_val = pre[metric].mean() if not pre.empty else np.nan
        post_val = post[metric].mean() if not post.empty else np.nan
        row[f'Pre_{metric}'] = pre_val
        row[f'Post_{metric}'] = post_val
        row[f'Abs_Change_{metric}'] = post_val - pre_val if pd.notna(pre_val) and pd.notna(post_val) else np.nan
        if metric in COUNT_METRICS:
            row[f'Pct_Change_{metric}'] = (
                (post_val - pre_val)/pre_val
                if pd.notna(pre_val) and pre_val != 0 and pd.notna(post_val)
                else np.nan
            )

    summary_rows.append(row)

    if idx % 100 == 0:
        print(f'Processed {idx:,} / {len(needed):,} location-date summaries')

loc_event = pd.DataFrame(summary_rows)
loc_event.to_csv(OUTPUT_DIR / '02_LOCATION_EVENT_BUYER_SUMMARY.csv', index=False)

print('Location-event summaries:', f'{len(loc_event):,}')
print('Median PRE periods:', loc_event['Pre_Periods'].median())
print('Median POST periods:', loc_event['Post_Periods'].median())


Processed 100 / 540 location-date summaries
Processed 300 / 540 location-date summaries
Processed 400 / 540 location-date summaries
Processed 500 / 540 location-date summaries
Location-event summaries: 525
Median PRE periods: 0.0
Median POST periods: 0.0


## 8. Attach treated and control buyer summaries to matched pairs

In [10]:

t = loc_event.rename(columns={'Location':'Treated_Location'}).copy()
t = t.rename(columns={c:f'T_{c}' for c in t.columns if c not in ['Treated_Location','Change_Date']})

c = loc_event.rename(columns={'Location':'Control_Location'}).copy()
c = c.rename(columns={x:f'C_{x}' for x in c.columns if x not in ['Control_Location','Change_Date']})

buyer_pair = pairs.merge(t, on=['Treated_Location','Change_Date'], how='left')
buyer_pair = buyer_pair.merge(c, on=['Control_Location','Change_Date'], how='left')

del t, c, loc_event
gc.collect()

for col in ['T_Pre_Periods','T_Post_Periods','C_Pre_Periods','C_Post_Periods']:
    buyer_pair[col] = pd.to_numeric(buyer_pair[col], errors='coerce').fillna(0)

# Create pair-level Buyer DiD columns before coverage filtering.
for metric in METRICS:
    buyer_pair[f'Buyer_DiD_Abs_{metric}'] = (
        buyer_pair[f'T_Abs_Change_{metric}'] - buyer_pair[f'C_Abs_Change_{metric}']
    )
    if metric in COUNT_METRICS:
        buyer_pair[f'Buyer_DiD_Pct_{metric}'] = (
            buyer_pair[f'T_Pct_Change_{metric}'] - buyer_pair[f'C_Pct_Change_{metric}']
        )

print('Matched pair rows with buyer summaries:', f'{len(buyer_pair):,}')


Matched pair rows with buyer summaries: 2,425


## 9. Coverage diagnostic and automatic best-rule selection

In [11]:

def coverage_mask(data, n):
    return (
        (data['T_Pre_Periods'] >= n)
        & (data['T_Post_Periods'] >= n)
        & (data['C_Pre_Periods'] >= n)
        & (data['C_Post_Periods'] >= n)
    )

m3 = coverage_mask(buyer_pair, 3)
m2 = coverage_mask(buyer_pair, 2)
m1 = coverage_mask(buyer_pair, 1)

n3, n2, n1 = int(m3.sum()), int(m2.sum()), int(m1.sum())

coverage_diag = pd.DataFrame({
    'Coverage_Rule':['3 PRE / 3 POST','2 PRE / 2 POST','1 PRE / 1 POST'],
    'Eligible_Pairs':[n3,n2,n1]
})

display(coverage_diag)

if n3 > 0:
    selected_mask = m3
    COVERAGE_RULE = '3 PRE / 3 POST'
    COVERAGE_QUALITY = 'Preferred'
elif n2 > 0:
    selected_mask = m2
    COVERAGE_RULE = '2 PRE / 2 POST'
    COVERAGE_QUALITY = 'Fallback'
elif n1 > 0:
    selected_mask = m1
    COVERAGE_RULE = '1 PRE / 1 POST'
    COVERAGE_QUALITY = 'Diagnostic Only'
else:
    raise ValueError(
        'No matched treated/control buyer pairs have even 1 PRE and 1 POST period. '
        'Review the printed date and location overlap diagnostics above.'
    )

eligible_pairs = buyer_pair.loc[selected_mask].copy()
eligible_pairs['Buyer_Coverage_Rule'] = COVERAGE_RULE
eligible_pairs['Buyer_Coverage_Quality'] = COVERAGE_QUALITY

coverage_diag['Selected'] = coverage_diag['Coverage_Rule'].eq(COVERAGE_RULE)
coverage_diag.to_csv(OUTPUT_DIR / '03_BUYER_COVERAGE_DIAGNOSTIC.csv', index=False)

print('Selected rule:', COVERAGE_RULE)
print('Quality:', COVERAGE_QUALITY)
print('Eligible pair rows:', f'{len(eligible_pairs):,}')
print('Eligible treated events:', eligible_pairs['Segment_Change_ID'].nunique())


,Coverage_Rule,Eligible_Pairs
0,3 PRE / 3 POST,0
1,2 PRE / 2 POST,0
2,1 PRE / 1 POST,0


ValueError: No matched treated/control buyer pairs have even 1 PRE and 1 POST period. Review the printed date and location overlap diagnostics above.

## 10. Collapse matched controls to one Buyer DiD row per treated event

In [12]:

id_cols = ['Segment_Change_ID','Corporate_Change_ID','Change_Date','BUYFEE_CHANGE_AMOUNT','Treated_Location']
for opt in ['Treated_Lane','Treated_Price_Tier','DiD_RPU_Change_Dollar','DiD_Conversion_Change_pp','DiD_Volume_Change_Pct']:
    if opt in eligible_pairs.columns:
        id_cols.append(opt)

agg_spec = {
    'Valid_Controls': ('Control_Rank','count'),
    'T_Pre_Periods': ('T_Pre_Periods','first'),
    'T_Post_Periods': ('T_Post_Periods','first'),
    'Min_Control_Pre_Periods': ('C_Pre_Periods','min'),
    'Min_Control_Post_Periods': ('C_Post_Periods','min'),
}

for metric in METRICS:
    agg_spec[f'Pre_{metric}'] = (f'T_Pre_{metric}','first')
    agg_spec[f'Buyer_DiD_Abs_{metric}'] = (f'Buyer_DiD_Abs_{metric}','mean')
    if metric in COUNT_METRICS:
        agg_spec[f'Buyer_DiD_Pct_{metric}'] = (f'Buyer_DiD_Pct_{metric}','mean')

buyer_event_did = (
    eligible_pairs.groupby(id_cols, dropna=False, as_index=False)
    .agg(**agg_spec)
)

buyer_event_did['Buyer_Coverage_Rule'] = COVERAGE_RULE
buyer_event_did['Buyer_Coverage_Quality'] = COVERAGE_QUALITY

buyer_event_did['RPU_Success'] = buyer_event_did['DiD_RPU_Change_Dollar'] > RPU_SUCCESS_MIN
buyer_event_did['Conversion_Safe'] = buyer_event_did['DiD_Conversion_Change_pp'] >= CONVERSION_SAFE_MIN_PP
buyer_event_did['Volume_Safe'] = buyer_event_did['DiD_Volume_Change_Pct'] >= VOLUME_SAFE_MIN_PCT
buyer_event_did['Balanced_Success'] = (
    buyer_event_did['RPU_Success']
    & buyer_event_did['Conversion_Safe']
    & buyer_event_did['Volume_Safe']
)

buyer_event_did.to_csv(OUTPUT_DIR / '04_BUYER_EVENT_LEVEL_DID.csv', index=False)

print('Buyer DiD events:', f'{len(buyer_event_did):,}')
print('Locations:', buyer_event_did['Treated_Location'].nunique())
print('Change dates:', buyer_event_did['Corporate_Change_ID'].nunique())
print('Median valid controls:', buyer_event_did['Valid_Controls'].median())


NameError: name 'eligible_pairs' is not defined

## 11. Scenario-level Buyer DiD summary

In [13]:

rows=[]
for fee in SCENARIOS:
    d = buyer_event_did[
        buyer_event_did['BUYFEE_CHANGE_AMOUNT'].between(
            fee - SCENARIO_TOLERANCE,
            fee + SCENARIO_TOLERANCE
        )
    ]
    row = {
        'Fee_Increase': fee,
        'Events': len(d),
        'Locations': d['Treated_Location'].nunique(),
        'Corporate_Change_Dates': d['Corporate_Change_ID'].nunique(),
        'Median_Valid_Controls': d['Valid_Controls'].median(),
    }
    for metric in COUNT_METRICS:
        row[f'Median_Buyer_DiD_Pct_{metric}'] = d[f'Buyer_DiD_Pct_{metric}'].median()
    for metric in ['Avg_Days_Since_Last_Purchase','Avg_Days_Since_Previous_Location_Purchase']:
        row[f'Median_Buyer_DiD_Days_{metric}'] = d[f'Buyer_DiD_Abs_{metric}'].median()
    rows.append(row)

scenario = pd.DataFrame(rows)
scenario.to_csv(OUTPUT_DIR / '05_BUYER_DID_BY_SCENARIO.csv', index=False)
display(scenario)


NameError: name 'buyer_event_did' is not defined

## 12. Buyer mechanism correlations

In [14]:

features = [f'Buyer_DiD_Pct_{m}' for m in COUNT_METRICS] + [
    'Buyer_DiD_Abs_Avg_Days_Since_Last_Purchase',
    'Buyer_DiD_Abs_Avg_Days_Since_Previous_Location_Purchase'
]
outcomes = ['DiD_RPU_Change_Dollar','DiD_Conversion_Change_pp','DiD_Volume_Change_Pct']

rows=[]
for feature in features:
    rec={'Buyer_Feature':feature}
    x = pd.to_numeric(buyer_event_did[feature], errors='coerce')
    rec['Events'] = int(x.notna().sum())
    for outcome in outcomes:
        y = pd.to_numeric(buyer_event_did[outcome], errors='coerce')
        mask = x.notna() & y.notna()
        rec[f'Corr_with_{outcome}'] = x[mask].corr(y[mask]) if mask.sum() >= 20 else np.nan
    rows.append(rec)

mechanism = pd.DataFrame(rows)
corr_cols = [c for c in mechanism.columns if c.startswith('Corr_with_')]
mechanism['Max_Absolute_Correlation'] = mechanism[corr_cols].abs().max(axis=1)
mechanism = mechanism.sort_values('Max_Absolute_Correlation', ascending=False)
mechanism.to_csv(OUTPUT_DIR / '06_BUYER_MECHANISM_CORRELATIONS.csv', index=False)
display(mechanism)


NameError: name 'buyer_event_did' is not defined

## 13. PRE-event buyer feature ranking

In [15]:

pre_features = [f'Pre_{m}' for m in METRICS]
rows=[]
for feature in pre_features:
    rec={'Feature':feature}
    x = pd.to_numeric(buyer_event_did[feature], errors='coerce')
    rec['Events'] = int(x.notna().sum())
    for outcome in outcomes:
        y = pd.to_numeric(buyer_event_did[outcome], errors='coerce')
        mask = x.notna() & y.notna()
        rec[f'Corr_with_{outcome}'] = x[mask].corr(y[mask]) if mask.sum() >= 20 else np.nan
    rows.append(rec)

ranking = pd.DataFrame(rows)
corr_cols = [c for c in ranking.columns if c.startswith('Corr_with_')]
ranking['Max_Absolute_Correlation'] = ranking[corr_cols].abs().max(axis=1)
ranking = ranking.sort_values('Max_Absolute_Correlation', ascending=False)
ranking.to_csv(OUTPUT_DIR / '07_PRE_BUYER_FEATURE_RANKING.csv', index=False)
display(ranking)


NameError: name 'buyer_event_did' is not defined

## 14. Final executive summary

In [16]:

summary = pd.DataFrame([
    {'Metric':'Raw_Buyer_Rows_Read','Value':rows_read},
    {'Metric':'Buyer_Biweekly_Aggregate_Rows','Value':len(buyer)},
    {'Metric':'Buyer_Locations','Value':buyer['VRLOCATION'].nunique()},
    {'Metric':'Matched_Pair_Rows','Value':len(buyer_pair)},
    {'Metric':'Eligible_Matched_Pairs','Value':len(eligible_pairs)},
    {'Metric':'Coverage_Rule_Used','Value':COVERAGE_RULE},
    {'Metric':'Coverage_Quality','Value':COVERAGE_QUALITY},
    {'Metric':'Buyer_DiD_Events','Value':len(buyer_event_did)},
    {'Metric':'Buyer_DiD_Locations','Value':buyer_event_did['Treated_Location'].nunique()},
    {'Metric':'Buyer_DiD_Change_Dates','Value':buyer_event_did['Corporate_Change_ID'].nunique()},
    {'Metric':'Median_Valid_Controls','Value':buyer_event_did['Valid_Controls'].median()},
])
summary.to_csv(OUTPUT_DIR / '08_V12A_3_EXECUTIVE_SUMMARY.csv', index=False)

print('=' * 80)
print('V12A.3 COMPLETE')
print('=' * 80)
print('Output folder:', OUTPUT_DIR)
display(summary)


NameError: name 'eligible_pairs' is not defined


## Files to share back

Please send:

1. `08_V12A_3_EXECUTIVE_SUMMARY.csv`
2. `05_BUYER_DID_BY_SCENARIO.csv`
3. `06_BUYER_MECHANISM_CORRELATIONS.csv`
4. `07_PRE_BUYER_FEATURE_RANKING.csv`
5. `04_BUYER_EVENT_LEVEL_DID.csv`
6. `03_BUYER_COVERAGE_DIAGNOSTIC.csv`

These outputs will tell us whether buyer behavior materially explains the +$10 / +$15 / +$20 fee-response heterogeneity and whether it should enter V12B.
